# Task 5 — Explainability (Grad-CAM + SHAP)
**COMP41840 AI for Health**  
**Owner:** Sergio  
**Goal:** Explain the imaging model via Grad-CAM and the tabular model via SHAP values

> **Prerequisite:** Run notebooks 02 and 03 first


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import shap
import cv2
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torchvision import transforms
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from monai.networks.nets import DenseNet121

sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT   = Path('../data/raw')
RESULTS_DIR = Path('../results')
(RESULTS_DIR / 'figures').mkdir(exist_ok=True)

---
## Part A — Grad-CAM for Imaging Model

### A.1 — Load trained imaging model

In [ ]:
model = DenseNet121(spatial_dims=2, in_channels=3, out_channels=2).to(DEVICE)
model.load_state_dict(torch.load(RESULTS_DIR / 'best_imaging_model.pt', map_location=DEVICE))
model.eval()
print('Model loaded.')

### A.2 — Grad-CAM Implementation

In [ ]:
class GradCAM:
    """Minimal Grad-CAM for a MONAI DenseNet121."""

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(dim=1).item()
        score = out[0, class_idx]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # GAP
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


# Hook into the last dense block's output
# For MONAI DenseNet121: model.features.denseblock4
target_layer = model.class_layers  # adjust based on MONAI version
# target_layer = list(model.features.children())[-2]  # alternative
grad_cam = GradCAM(model, target_layer)

### A.3 — Visualise Grad-CAM for Sample Images

In [ ]:
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

def visualise_gradcam(img_path, label_str, save_path=None):
    raw_img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    tensor  = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    tensor.requires_grad_()

    cam, pred_cls = grad_cam(tensor)
    heatmap = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    heatmap_color = cm.jet(heatmap)[:, :, :3]
    overlay = 0.5 * np.array(raw_img) / 255 + 0.5 * heatmap_color

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_img, cmap='gray'); axes[0].set_title(f'Original ({label_str})')
    axes[1].imshow(heatmap, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap')
    axes[2].imshow(overlay); axes[2].set_title(f'Overlay | Pred: {["benign","malignant"][pred_cls]}')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


# Visualise one malignant and one benign example
mal_sample = sorted((DATA_ROOT / 'malignant' / 'images').glob('*.png'))[0]
ben_sample = sorted((DATA_ROOT / 'benign'    / 'images').glob('*.png'))[0]

visualise_gradcam(mal_sample, 'malignant', RESULTS_DIR / 'figures/gradcam_malignant.png')
visualise_gradcam(ben_sample, 'benign',    RESULTS_DIR / 'figures/gradcam_benign.png')

### A.4 — Grad-CAM Discussion

Describe what regions the model attends to:
- For malignant cases: does the heatmap centre on the tumour mass?
- For benign cases: are the highlighted regions clinically plausible?
- Discuss any surprising or concerning attention patterns.

---
## Part B — SHAP Values for Tabular Model

### B.1 — Load Trained XGBoost Model & Preprocess

In [ ]:
# Reload tabular data — replicate exact preprocessing from notebook 03
df = pd.read_csv(DATA_ROOT.parent / 'clinical.csv')
df_binary = df[df['label'].isin(['benign', 'malignant'])].copy()
df_binary['label_enc'] = (df_binary['label'] == 'malignant').astype(int)

TARGET   = 'label_enc'
DROP_COLS = ['label', 'patient_id', 'image_path']  # adjust as needed
feature_cols = [c for c in df_binary.columns if c not in DROP_COLS + [TARGET]]

X = df_binary[feature_cols].copy()
y = df_binary[TARGET].values

cat_cols = X.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)
X_imp_df = pd.DataFrame(X_imp, columns=feature_cols)

# Reload model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(str(RESULTS_DIR / 'xgb_model.json'))  # save in notebook 03 if not already done
print('XGBoost model loaded.')

### B.2 — SHAP Summary (Beeswarm) Plot

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_imp_df)

# shap_values may be a list [class0, class1] — use class 1 (malignant)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_imp_df, show=False)
plt.title('SHAP Summary — Feature Impact on Malignant Prediction')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar summary — mean absolute SHAP
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_imp_df, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### B.3 — SHAP Waterfall for Individual Patients

In [ ]:
# Waterfall for one malignant and one benign patient
shap_exp = explainer(X_imp_df)

malignant_idxs = np.where(y == 1)[0]
benign_idxs    = np.where(y == 0)[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.sca(axes[0])
shap.plots.waterfall(shap_exp[malignant_idxs[0]], show=False)
axes[0].set_title('Malignant patient')

plt.sca(axes[1])
shap.plots.waterfall(shap_exp[benign_idxs[0]], show=False)
axes[1].set_title('Benign patient')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

### B.4 — SHAP Dependence Plots for Top Features

In [ ]:
# Top 2 most impactful features
mean_abs = np.abs(sv).mean(0)
top2 = np.argsort(mean_abs)[-2:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, feat_idx in zip(axes, top2):
    feat_name = feature_cols[feat_idx]
    shap.dependence_plot(feat_name, sv, X_imp_df, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence: {feat_name}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

### B.5 — Explainability Discussion

Answer the following questions in the report:

1. **Grad-CAM:** Do the highlighted image regions align with known ultrasound indicators of malignancy (e.g., irregular margins, posterior shadowing)? Are there spurious correlations?
2. **SHAP:** Which clinical features drive malignant predictions? Are they consistent with domain knowledge?
3. **Plausibility:** Do both methods provide clinically plausible explanations, or do they reveal model artefacts?
4. **Limitations:** What are the limits of Grad-CAM and SHAP in a medical context?